# Tahap 2 — Dataset Exploration

Notebook ini menyajikan bukti eksplorasi dataset dan preprocessing resmi. Perhitungan tetap berada di `src/preprocessing/`; notebook tidak mendefinisikan statistik alternatif dan tidak mengubah raw data.

In [ ]:
# 2. Environment check
from pathlib import Path
import importlib.metadata
import os
import platform
import sys

EXPECTED_DATASET_SHA256 = 'ca7831a188a191edbf82a673fac90dbb875b5095986ed07699c02530f2a02a0e'
REPOSITORY_URL = 'https://github.com/rehanalfarizu/new_jurnal.git'
IS_COLAB = 'google.colab' in sys.modules

def find_repository_root():
    candidates = [Path.cwd(), Path.cwd() / 'new_jurnal', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'configs' / 'experiment.yaml').is_file() and (candidate / 'src').is_dir():
            return candidate.resolve()
    return None

REPO_ROOT = find_repository_root()
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'colab': IS_COLAB, 'repository_found': REPO_ROOT is not None})

In [ ]:
# 3. Google Colab setup
import subprocess

if IS_COLAB and REPO_ROOT is None:
    clone_target = Path('/content/new_jurnal')
    if not clone_target.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(clone_target)], check=True)
    REPO_ROOT = clone_target.resolve()
elif REPO_ROOT is None:
    raise RuntimeError('Repository tidak ditemukan. Jalankan notebook dari root new_jurnal atau direktori notebooks/.')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository aktif:', REPO_ROOT.name)

In [ ]:
# 4. Dependency setup
import importlib.util

if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
required_modules = {'pandas': 'pandas', 'PyYAML': 'yaml'}
missing = [name for name, module in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError('Dependency kernel belum lengkap: ' + ', '.join(missing))
print('Dependency requirements.txt tersedia pada kernel aktif.')

In [ ]:
# 5. SENSOR_DATA_PATH
# Google Colab opsional:
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ['SENSOR_DATA_PATH'] = '/content/drive/MyDrive/.../sensor_data.csv'
configured_path = os.environ.get('SENSOR_DATA_PATH', '').strip()
default_path = (REPO_ROOT / 'data/raw/sensor_data.csv').resolve()
if configured_path:
    DATASET_PATH = Path(configured_path).expanduser()
    if not DATASET_PATH.is_absolute():
        DATASET_PATH = (REPO_ROOT / DATASET_PATH).resolve()
    path_source = 'SENSOR_DATA_PATH'
elif default_path.is_file():
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'
elif not IS_COLAB:
    candidates_by_file = {}
    for pattern in ('*/Data/sensor_data.csv', '*/data/sensor_data.csv'):
        for path in REPO_ROOT.parent.glob(pattern):
            if path.is_file():
                info = path.stat()
                candidates_by_file[(info.st_dev, info.st_ino)] = path.resolve()
    candidates = sorted(candidates_by_file.values())
    if len(candidates) > 1:
        raise RuntimeError('Lebih dari satu dataset ditemukan; tetapkan SENSOR_DATA_PATH secara eksplisit.')
    DATASET_PATH = candidates[0] if candidates else default_path
    path_source = 'kandidat tunggal folder proyek saudara' if candidates else 'data/raw/sensor_data.csv'
else:
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'
if not DATASET_PATH.is_file():
    raise FileNotFoundError('sensor_data.csv tidak ditemukan. Tetapkan SENSOR_DATA_PATH atau mount Google Drive.')
print('Sumber resolusi dataset:', path_source)

In [ ]:
# 6. SHA-256 dataset verification
import hashlib

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

DATASET_SHA256 = sha256_file(DATASET_PATH)
if DATASET_SHA256 != EXPECTED_DATASET_SHA256:
    raise ValueError(f'Checksum dataset tidak sesuai: {DATASET_SHA256}')
print('SHA-256 terverifikasi:', DATASET_SHA256)

In [ ]:
# 7. Dataset schema
import pandas as pd
from IPython.display import display
from src.preprocessing.pipeline import analyze_dataset, load_stage2_config

stage2_config = load_stage2_config('configs/experiment.yaml')
column_schema = pd.read_csv('results/tables/column_schema.csv')
display(column_schema)
print('Pipeline source of truth:', analyze_dataset.__module__ + '.' + analyze_dataset.__name__)

In [ ]:
# 8. Record count dan observation period
dataset_summary = pd.read_csv('results/tables/dataset_summary.csv')
official_sha = str(dataset_summary.loc[dataset_summary['metrik'] == 'sha256', 'nilai'].iloc[0])
if official_sha != DATASET_SHA256:
    raise AssertionError('Checksum tabel resmi tidak cocok dengan dataset aktif.')
display(dataset_summary[dataset_summary['metrik'].isin(['jumlah_record', 'jumlah_kolom', 'waktu_awal_utc', 'waktu_akhir_utc', 'durasi_observasi_detik', 'jumlah_device'])])

In [ ]:
# 9. UTC handling
preprocessing_report = pd.read_csv('results/tables/preprocessing_report.csv')
display(preprocessing_report[preprocessing_report['transformasi'].isin(['parse_timestamp', 'normalisasi_utc', 'urut_kronologis'])])
print('Source timezone:', stage2_config['data']['source_timezone'])
print('Policy:', stage2_config['preprocessing']['timezone_policy'])
print('Timestamp naive diinterpretasikan sebagai UTC tanpa mengubah clock value.')

In [ ]:
# 10. Missing values
missing_values = pd.read_csv('results/tables/missing_values.csv')
display(missing_values)

In [ ]:
# 11. Exact duplicate dan duplicate timestamp
duplicate_summary = pd.read_csv('results/tables/duplicate_summary.csv')
display(duplicate_summary)
print('Kebijakan resmi mempertahankan data dan melaporkan duplikasi; tidak ada penghapusan diam-diam.')

In [ ]:
# 12. Sampling interval distribution
sampling_intervals = pd.read_csv('results/tables/sampling_interval_distribution.csv')
display(sampling_intervals)

In [ ]:
# 13. Temporal gaps
temporal_gaps = pd.read_csv('results/tables/temporal_gaps.csv')
display(temporal_gaps.head(20))
print('Jumlah gap resmi:', len(temporal_gaps), '| threshold detik:', stage2_config['preprocessing']['gap_threshold_seconds'])

In [ ]:
# 14. Descriptive statistics
descriptive_statistics = pd.read_csv('results/tables/descriptive_statistics.csv')
display(descriptive_statistics)

In [ ]:
# 15. Variable distribution / exploratory outlier summary
variable_distributions = pd.read_csv('results/tables/variable_distributions.csv')
outlier_summary = pd.read_csv('results/tables/outlier_summary.csv')
display(variable_distributions.groupby('variable', sort=False).head(5))
display(outlier_summary)
print('IQR hanya indikator eksploratif; seluruh nilai dipertahankan dan dilaporkan.')

In [ ]:
# 16. Preprocessing summary
display(preprocessing_report)
if not (preprocessing_report['record_sebelum'] == preprocessing_report['record_setelah']).all():
    raise AssertionError('Ditemukan perubahan jumlah record pada laporan preprocessing.')

In [ ]:
# 17. Limitations
from IPython.display import Markdown
display(Markdown('''### Keterbatasan

- Dataset berasal dari satu gateway dan satu periode observasi.
- Timestamp kamera, sensor, gateway arrival, dan cloud ingestion tidak tersedia secara independen.
- Occupancy menggunakan latest available camera snapshot; nearest-timestamp synchronization tidak dapat dibuktikan.
- Outlier IQR tidak otomatis berarti measurement error dan tidak dihapus.
- Eksplorasi ini tidak membuktikan generalisasi, latency, atau efek kausal.'''))

In [ ]:
# 18. Reproducibility summary
official_outputs = [
    'dataset_summary.csv', 'column_schema.csv', 'missing_values.csv',
    'duplicate_summary.csv', 'sampling_interval_distribution.csv',
    'temporal_gaps.csv', 'descriptive_statistics.csv',
    'variable_distributions.csv', 'outlier_summary.csv', 'preprocessing_report.csv',
]
summary = {
    'dataset_sha256': DATASET_SHA256,
    'config': 'configs/experiment.yaml',
    'pipeline': analyze_dataset.__module__ + '.' + analyze_dataset.__name__,
    'official_outputs_available': all((Path('results/tables') / name).is_file() for name in official_outputs),
    'statistics_recomputed_in_notebook': False,
}
display(summary)